# 02 · Limiar de confiança e comparação entre o app e o Python

Este notebook usa o modelo do experimento `yolo26n_cls_v1`, treinado no notebook 01, para duas tarefas:

1. **Limiar de confiança baixa:** escolher, só com as imagens de validação, a confiança abaixo da qual o app pede uma nova foto, e medir o efeito na partição de teste.
2. **Comparação entre o app e o Python:** classificar no aparelho as imagens de teste, pelo mesmo caminho do app, e conferir se o resultado bate com o do `ultralytics`.

Ele ocupa o lugar do notebook que avaliaria as fotos próprias, cancelado em 16/11/2026.

**Integridade:** os números que vão para a monografia saem das execuções deste notebook e ficam em `ml/resultados/yolo26n_cls_v1/` (`predicoes_tflite.csv`, `limiar.json`, `paridade_app.csv` e `paridade.json`).

Roda no Google Colab ou num computador com Python 3.13; a CPU basta.

Executado em 16/11/2026 fora do Colab, num Mac com CPU Apple M1 (sem GPU). Nas saídas, os caminhos locais foram trocados por `<TCC_DADOS>`, `<TCC_MODELOS>` e `<PASTA_DE_TRABALHO>`.

## 1. Instalação e caminhos

In [1]:
import importlib.util, os, subprocess, sys
from pathlib import Path

if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.188', 'huggingface_hub', 'scikit-learn'], check=True)
import torch, ultralytics
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| python', sys.version.split()[0])

SEED = 42
DATASET_REPO = 'ParsaVictor/tomato-ripeness-detection'
DATASET_REV = '91481266b3674f4eec31e72310888321d0e102f2'  # mesma revisão do notebook 01
NOME_EXPERIMENTO = 'yolo26n_cls_v1'
MAPA_CLASSES = {'raw': 'verde', 'half_ripe': 'de_vez', 'ripe': 'maduro', 'rotten': 'passado'}
LIMITE_OUTRO = {'train': 4000, 'val': 700, 'test': 150}

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/tcc-maturacao')
    MODELOS = DRIVE / 'runs' / NOME_EXPERIMENTO / 'weights'   # best.pt e best.tflite do notebook 01
    REFERENCIA = DRIVE / 'runs' / NOME_EXPERIMENTO / 'metricas_teste.json'
    RESULTADOS = DRIVE / 'resultados' / NOME_EXPERIMENTO
    DATA = Path('/content/data')
    ZIPS = None  # baixa do Hugging Face
else:
    # Fora do Colab, as pastas vêm de variáveis de ambiente.
    MODELOS = Path(os.environ['TCC_MODELOS'])           # pasta com best.pt e best.tflite
    RESULTADOS = Path(os.environ['TCC_RESULTADOS'])     # ml/resultados/yolo26n_cls_v1 do repositório
    REFERENCIA = RESULTADOS / 'metricas_teste.json'
    DATA = Path(os.environ['TCC_DADOS'])                # pasta de trabalho, com uns 3 GB livres
    ZIPS = Path(os.environ['TCC_ZIPS']) if 'TCC_ZIPS' in os.environ else None  # zips já baixados
RESULTADOS.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
DATASET = DATA / 'tomate_cls'

ultralytics 8.4.188 | torch 2.14.1 | python 3.13.15


## 2. Dados: as mesmas pastas do notebook 01

Repete as seções 3 a 5 do notebook 01, com a mesma revisão do dataset e a mesma semente, para recriar os recortes da classe outro iguais aos do treino. Se as pastas já existirem, esta célula só as conta.

In [2]:
import shutil, zipfile
import numpy as np
from PIL import Image
def baixar(nome):
    if ZIPS:
        return ZIPS / nome
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(DATASET_REPO, nome, repo_type='dataset', revision=DATASET_REV))

def contar(raiz):
    for split in ('train', 'val', 'test'):
        pastas = sorted(p for p in (raiz / split).iterdir() if p.is_dir())
        print(split, {p.name: sum(1 for _ in p.iterdir()) for p in pastas})

def caixas_em_pixels(label_path, largura, altura):
    caixas = []
    if label_path.exists():
        for linha in label_path.read_text().splitlines():
            partes = linha.split()
            if len(partes) >= 5:
                _, xc, yc, bw, bh = map(float, partes[:5])
                caixas.append(((xc - bw / 2) * largura, (yc - bh / 2) * altura,
                               (xc + bw / 2) * largura, (yc + bh / 2) * altura))
    return caixas

def sobrepoe(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

if not (DATASET / 'test' / 'outro').exists():
    with zipfile.ZipFile(baixar('tomato_ripeness_crops.zip')) as z:
        z.extractall(DATA / 'crops_raw')
    origem = next((DATA / 'crops_raw').rglob('cls_tomato'))
    if DATASET.exists():
        shutil.rmtree(DATASET)
    for split in ('train', 'val', 'test'):
        for original, nova in MAPA_CLASSES.items():
            shutil.copytree(origem / split / original, DATASET / split / nova)

    with zipfile.ZipFile(baixar('tomato_unified_detection.zip')) as z:
        z.extractall(DATA / 'det_raw')
    unified = next((DATA / 'det_raw').rglob('unified'))
    rng = np.random.default_rng(SEED)
    for split, pasta in {'train': 'train', 'val': 'valid', 'test': 'test'}.items():
        imagens = sorted(p for p in unified.glob(f'*/{pasta}/images/*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        rng.shuffle(imagens)
        destino = DATASET / split / 'outro'
        destino.mkdir(parents=True, exist_ok=True)
        feitos = 0
        for img_path in imagens:
            if feitos >= LIMITE_OUTRO[split]:
                break
            label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
            with Image.open(img_path) as im:
                im = im.convert('RGB')
                largura, altura = im.size
                caixas = caixas_em_pixels(label_path, largura, altura)
                lado = int(min(largura, altura) * rng.uniform(0.15, 0.35))
                for _ in range(20):
                    x0 = int(rng.integers(0, largura - lado))
                    y0 = int(rng.integers(0, altura - lado))
                    recorte = (x0, y0, x0 + lado, y0 + lado)
                    if not any(sobrepoe(recorte, c) for c in caixas):
                        im.crop(recorte).resize((160, 160)).save(destino / f'outro_{split}_{feitos:05d}.jpg', quality=92)
                        feitos += 1
                        break
        print(f'{split}: {feitos} recortes "outro" de {len(imagens)} imagens disponíveis')
contar(DATASET)

train {'de_vez': 10827, 'maduro': 10540, 'outro': 4000, 'passado': 4820, 'verde': 24589}
val {'de_vez': 1611, 'maduro': 2180, 'outro': 700, 'passado': 80, 'verde': 4708}
test {'de_vez': 339, 'maduro': 196, 'outro': 150, 'passado': 47, 'verde': 186}


## 3. Conferência com o notebook 01

O teste refeito aqui, com os pesos `best.pt`, precisa dar a mesma acurácia registrada no `metricas_teste.json` do notebook 01. Se der, as pastas são as mesmas do treino.

In [3]:
import json
from ultralytics import YOLO

modelo_pt = YOLO(str(MODELOS / 'best.pt'))
metricas = modelo_pt.val(data=str(DATASET), split='test', imgsz=224, device='cpu', plots=False, verbose=False)
referencia = json.loads(REFERENCIA.read_text())
print('top-1 no teste, aqui:      ', round(float(metricas.top1), 4))
print('top-1 no teste, notebook 01:', round(referencia['top1_ultralytics'], 4))

Ultralytics 8.4.188 🚀 Python-3.13.15 torch-2.14.1 CPU (Apple M1)


YOLO26n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs


train: <TCC_DADOS>/tomate_cls/train... found 54776 images in 5 classes ✅ 


val: <TCC_DADOS>/tomate_cls/val... found 9279 images in 5 classes ✅ 


test: <TCC_DADOS>/tomate_cls/test... found 918 images in 5 classes ✅ 


test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 10.9±5.4 MB/s, size: 5.0 KB)


... (49 outputs left out)

               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 58/58 9.2it/s 6.3s

                   all      0.849          1


Speed: 0.0ms preprocess, 4.7ms inference, 0.0ms loss, 0.0ms postprocess per image


top-1 no teste, aqui:       0.8486
top-1 no teste, notebook 01: 0.8486


## 4. Previsões do `.tflite`, imagem a imagem

O app usa o `.tflite`, então as confianças do limiar saem dele. A tabela grava a classe prevista, a confiança e a probabilidade de cada classe para todas as imagens de validação e de teste.

In [4]:
import csv
from sklearn.metrics import classification_report, confusion_matrix

modelo_tflite = YOLO(str(MODELOS / 'best.tflite'), task='classify')
arquivo_predicoes = RESULTADOS / 'predicoes_tflite.csv'
with open(arquivo_predicoes, 'w', newline='') as f:
    escritor = None
    for split in ('val', 'test'):
        for pasta in sorted(p for p in (DATASET / split).iterdir() if p.is_dir()):
            for arquivo in sorted(pasta.iterdir()):
                r = modelo_tflite.predict(str(arquivo), imgsz=224, device='cpu', verbose=False)[0]
                nomes = r.names
                if escritor is None:
                    escritor = csv.writer(f)
                    escritor.writerow(['particao', 'arquivo', 'classe_real', 'classe_prevista', 'confianca'] + [nomes[i] for i in sorted(nomes)])
                escritor.writerow([split, arquivo.name, pasta.name, nomes[int(r.probs.top1)], f'{float(r.probs.top1conf):.6f}']
                                  + [f'{v:.6f}' for v in r.probs.data.tolist()])

predicoes = list(csv.DictReader(open(arquivo_predicoes)))
teste = [p for p in predicoes if p['particao'] == 'test']
rotulos = sorted({p['classe_real'] for p in teste})
print(classification_report([p['classe_real'] for p in teste], [p['classe_prevista'] for p in teste], labels=rotulos, digits=3))
print(confusion_matrix([p['classe_real'] for p in teste], [p['classe_prevista'] for p in teste], labels=rotulos))

Loading <TCC_MODELOS>/best.tflite for LiteRT inference...


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


              precision    recall  f1-score   support

      de_vez      0.797     0.855     0.825       339
      maduro      0.789     0.781     0.785       196
       outro      1.000     1.000     1.000       150
     passado      0.848     0.830     0.839        47
       verde      0.896     0.790     0.840       186

    accuracy                          0.849       918
   macro avg      0.866     0.851     0.858       918
weighted avg      0.851     0.849     0.849       918

[[290  33   0   0  16]
 [ 35 153   0   7   1]
 [  0   0 150   0   0]
 [  1   7   0  39   0]
 [ 38   1   0   0 147]]


## 5. Limiar de confiança

**Regra, fixada antes de olhar o teste:** o limiar é o maior valor, entre 0,20 e 0,99 (de 0,01 em 0,01), em que as imagens de **validação** com confiança abaixo dele erram mais do que acertam (acurácia abaixo de 50%). A ideia é pedir uma nova foto só quando a resposta do modelo tem mais chance de estar errada do que certa.

Depois, o mesmo limiar é aplicado ao teste, para medir quantas imagens o app aceitaria e quantos erros ele barraria.

In [5]:
def dados(particao):
    return [(float(p['confianca']), p['classe_prevista'] == p['classe_real']) for p in predicoes if p['particao'] == particao]

def medir(d, t):
    aceitas = [ok for conf, ok in d if conf >= t]
    rejeitadas = [ok for conf, ok in d if conf < t]
    return {'limiar': round(t, 2), 'imagens': len(d), 'aceitas': len(aceitas),
            'fracao_aceitas': len(aceitas) / len(d),
            'acuracia_aceitas': sum(aceitas) / len(aceitas) if aceitas else None,
            'rejeitadas': len(rejeitadas), 'erros_rejeitados': sum(1 for ok in rejeitadas if not ok),
            'acertos_rejeitados': sum(1 for ok in rejeitadas if ok),
            'acuracia_rejeitadas': sum(rejeitadas) / len(rejeitadas) if rejeitadas else None}

val, tst = dados('val'), dados('test')
grade = [round(0.20 + 0.01 * i, 2) for i in range(80)]
validos = [t for t in grade if medir(val, t)['rejeitadas'] > 0 and medir(val, t)['acuracia_rejeitadas'] < 0.5]
limiar = max(validos)
print('limiar escolhido na validação:', limiar)
print()
print('limiar | validação: aceitas, acurácia das aceitas | teste: aceitas, acurácia das aceitas, erros barrados')
for t in sorted({0.5, 0.6, 0.7, 0.8, 0.9, limiar}):
    mv, mt = medir(val, t), medir(tst, t)
    print(f"{t:.2f}   | {mv['fracao_aceitas']:.3f}, {mv['acuracia_aceitas']:.3f}"
          f" | {mt['aceitas']}/{mt['imagens']} ({mt['fracao_aceitas']:.3f}), {mt['acuracia_aceitas']:.3f}, {mt['erros_rejeitados']} de {mt['erros_rejeitados'] + sum(1 for c, ok in tst if c >= t and not ok)}")

resultado_limiar = {
    'regra': 'maior limiar, de 0,20 a 0,99, em que as imagens de validação abaixo dele têm acurácia abaixo de 50%',
    'limiar': limiar,
    'validacao': medir(val, limiar),
    'teste': medir(tst, limiar),
    'teste_com_limiar_provisorio_0_5': medir(tst, 0.5),
}
(RESULTADOS / 'limiar.json').write_text(json.dumps(resultado_limiar, indent=2, ensure_ascii=False))
print(json.dumps(resultado_limiar, indent=2, ensure_ascii=False))

limiar escolhido na validação:

 0.47

limiar | validação: aceitas, acurácia das aceitas | teste: aceitas, acurácia das aceitas, erros barrados
0.47   | 0.999, 0.961 | 916/918 (0.998), 0.849, 1 de 139
0.50   | 0.998, 0.961 | 915/918 (0.997), 0.850, 2 de 139
0.60   | 0.981, 0.967 | 851/918 (0.927), 0.877, 34 de 139
0.70   | 0.962, 0.975 | 797/918 (0.868), 0.895, 55 de 139
0.80   | 0.938, 0.981 | 720/918 (0.784), 0.918, 80 de 139
0.90   | 0.901, 0.987 | 595/918 (0.648), 0.939, 103 de 139
{
  "regra": "maior limiar, de 0,20 a 0,99, em que as imagens de validação abaixo dele têm acurácia abaixo de 50%",
  "limiar": 0.47,
  "validacao": {
    "limiar": 0.47,
    "imagens": 9279,
    "aceitas": 9270,
    "fracao_aceitas": 0.9990300678952473,
    "acuracia_aceitas": 0.9607335490830636,
    "rejeitadas": 9,
    "erros_rejeitados": 5,
    "acertos_rejeitados": 4,
    "acuracia_rejeitadas": 0.4444444444444444
  },
  "teste": {
    "limiar": 0.47,
    "imagens": 918,
    "aceitas": 916,
    "fracao_aceitas": 0.9978213507625272,

## 6. Comparação entre o app e o Python

O teste `app/integration_test/paridade_test.dart` classifica no aparelho cada imagem de teste pelo mesmo caminho da tela de conferência do app: limpa a foto (`limparFoto`), recorta o quadrado central, redimensiona para 224 *pixels* e roda o `.tflite` com o `flutter_litert`. Para rodar, num emulador ou celular Android, com o app já instalado em modo debug (por exemplo, com `flutter run`):

```bash
COPYFILE_DISABLE=1 tar -C <DATA>/tomate_cls/test -cf - . | adb exec-in run-as br.tcc.maturacao_tomate sh -c 'rm -rf files/paridade && mkdir -p files/paridade && tar -xf - -C files/paridade'
cd app && flutter test integration_test/paridade_test.dart --no-uninstall
adb exec-out run-as br.tcc.maturacao_tomate cat files/paridade_app.csv > ../ml/resultados/yolo26n_cls_v1/paridade_app.csv
```

As imagens vão para a pasta interna do app porque ele não lê pastas criadas pelo `adb` em `Android/data`. O `--no-uninstall` evita que o `flutter test` desinstale o app no fim, o que apagaria as imagens.

A célula abaixo compara esse CSV com as previsões do `.tflite` no Python (seção 4).

A primeira versão do app ampliava as imagens menores que a entrada do modelo pela média de área do pacote `image`, que, na ampliação, equivale ao vizinho mais próximo; o `ultralytics` usa a interpolação bilinear. O resultado daquela versão ficou guardado em `paridade_app_media_de_area.csv`, e a célula compara as duas versões.

In [6]:
import statistics as st

def comparar(arquivo_app):
    py = {(p['classe_real'], p['arquivo']): p for p in teste}
    app = {tuple(r['arquivo'].split('/', 1)): r for r in csv.DictReader(open(arquivo_app))}
    chaves = sorted(set(py) & set(app))
    classes = sorted(MAPA_CLASSES.values()) + ['outro']
    iguais = [k for k in chaves if py[k]['classe_prevista'] == app[k]['classe_prevista']]
    dif_conf = [abs(float(py[k]['confianca']) - float(app[k]['confianca'])) for k in chaves]
    dif_max = [max(abs(float(py[k][c]) - float(app[k][c])) for c in classes) for k in chaves]
    return {
        'imagens': len(chaves),
        'mesma_classe': len(iguais),
        'fracao_mesma_classe': len(iguais) / len(chaves),
        'acertos_app': sum(1 for k in chaves if app[k]['classe_prevista'] == k[0]),
        'acertos_python': sum(1 for k in chaves if py[k]['classe_prevista'] == k[0]),
        'diferenca_confianca_mediana': st.median(dif_conf),
        'diferenca_confianca_media': st.mean(dif_conf),
        'maior_diferenca_probabilidade': max(dif_max),
        'tempo_inferencia_ms_mediana': st.median(float(app[k]['tempo_ms']) for k in chaves),
        'divergencias': [{'classe_real': k[0], 'arquivo': k[1],
                          'python': py[k]['classe_prevista'], 'confianca_python': float(py[k]['confianca']),
                          'app': app[k]['classe_prevista'], 'confianca_app': float(app[k]['confianca'])}
                         for k in chaves if py[k]['classe_prevista'] != app[k]['classe_prevista']],
    }

arquivo_app = RESULTADOS / 'paridade_app.csv'
if not arquivo_app.exists():
    print('Falta o', arquivo_app.name, '- rode o teste do app (instruções acima).')
else:
    paridade = {'app_atual': comparar(arquivo_app)}
    anterior = RESULTADOS / 'paridade_app_media_de_area.csv'
    if anterior.exists():
        paridade['app_com_media_de_area'] = comparar(anterior)
    (RESULTADOS / 'paridade.json').write_text(json.dumps(paridade, indent=2, ensure_ascii=False))
    for versao, r in paridade.items():
        print(versao, {k: v for k, v in r.items() if k != 'divergencias'})
        print('  divergências:', len(r['divergencias']))

app_atual {'imagens': 918, 'mesma_classe': 897, 'fracao_mesma_classe': 0.9771241830065359, 'acertos_app': 783, 'acertos_python': 779, 'diferenca_confianca_mediana': 0.004300999999999999, 'diferenca_confianca_media': 0.01596256209150327, 'maior_diferenca_probabilidade': 0.332244, 'tempo_inferencia_ms_mediana': 64.75}
  divergências: 21
app_com_media_de_area {'imagens': 918, 'mesma_classe': 879, 'fracao_mesma_classe': 0.9575163398692811, 'acertos_app': 778, 'acertos_python': 779, 'diferenca_confianca_mediana': 0.008495500000000045, 'diferenca_confianca_media': 0.027823209150326798, 'maior_diferenca_probabilidade': 0.998833, 'tempo_inferencia_ms_mediana': 64.80000000000001}
  divergências: 39
